In [0]:
%sql
DROP TABLE IF EXISTS catalog_ventas.silver.insert_;

CREATE TABLE catalog_ventas.silver.insert_(
  venta BIGINT,
  articulo INT,
  descrip STRING, 
  precio DECIMAL(15,2), 
  cant INT, 
  total DECIMAL(15,2), 
  usulogin STRING, 
  clientes STRING, 
  vtaestado STRING, 
  vtafecha TIMESTAMP, 
  condvtapos STRING, 
  delivery BOOLEAN,
  turno STRING, 
  _processing_timestamp TIMESTAMP DEFAULT CURRENT_TIMESTAMP()
)
USING DELTA
TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported')
COMMENT' Tabla de practica - insert o'

In [0]:
%sql
-- Primer carga 
MERGE INTO catalog_ventas.silver.insert_ AS target 
USING (
SELECT
  venta,
  articulo,
  descrip, 
  precio,
  cant ,
  total, 
  usulogin, 
  clientes, 
  vtaestado,
  vtafecha, 
  condvtapos, 
  delivery, 
  turno, 
  CURRENT_TIMESTAMP() AS _processing_timestamp 
FROM catalog_ventas.silver.ventas_clean 
WHERE vtaestado = 'normal'
LIMIT 2999
) AS source 
ON target.venta = source.venta
WHEN NOT MATCHED THEN INSERT *

    

    

In [0]:
%sql
SELECT 
  vtaestado,
  COUNT(*) AS total_registros
FROM catalog_ventas.silver.insert_
GROUP BY vtaestado

In [0]:
%sql
INSERT INTO catalog_ventas.silver.insert_
SELECT
  venta,
  articulo,
  descrip, 
  precio,
  cant ,
  total, 
  usulogin, 
  clientes, 
  vtaestado,
  vtafecha, 
  condvtapos, 
  delivery, 
  turno, 
  CURRENT_TIMESTAMP() AS _processing_timestamp 
FROM catalog_ventas.silver.ventas_clean
WHERE vtaestado = 'normal'
LIMIT 1000

In [0]:
%sql
INSERT INTO catalog_ventas.silver.insert_overwrite
SELECT
  venta,
  articulo,
  descrip, 
  precio,
  cant ,
  total, 
  usulogin, 
  clientes, 
  vtaestado,
  vtafecha, 
  condvtapos, 
  delivery, 
  turno, 
  CURRENT_TIMESTAMP() AS _processing_timestamp 
FROM catalog_ventas.silver.ventas_clean v
WHERE vtaestado = 'anulado'
AND NOT EXISTS(
  SELECT 1
  FROM catalog_ventas.silver.insert_overwrite iow
  WHERE iow.venta = v.venta
  AND iow.articulo = v.articulo
)
LIMIT 200

In [0]:
%sql
select 
venta,
articulo,
descrip,
count(*) as total
from catalog_ventas.silver.insert_ 
group by articulo, venta,descrip
having count(*) > 1

In [0]:
%sql
select 
  'clientes' AS columna,
  count(*) as total_registros,
  try_Cast(count(distinct cliente) AS BIGINT) as valores_unicos,
  round((valores_unicos * 100.00/ total_registros),2) as pct_cardinalidad
from catalog_ventas.silver.ventas_clean
group by cliente

union all

select 
  'articulo' AS columna,
  count(*) as total_registros,
  count(distinct articulo) as valores_unicos,
  round((valores_unicos * 100.00/ total_registros),2) as pct_cardinalidad
from catalog_ventas.silver.ventas_clean
group by articulo

union all

select 
  'condvtapos' AS columna,
  count(*) as total_registros,
  try_cast(count(distinct medio_pago) as bigint) as valores_unicos,
  round((valores_unicos * 100.00/ total_registros),2) as pct_cardinalidad
from catalog_ventas.silver.ventas_clean
group by medio_pago
order by columna

In [0]:
%sql
WITH base AS (

SELECT
    articulo,
    descrip,
    precio,
    cliente,
    CASE

        WHEN LOWER(descrip) LIKE '%torta%' THEN 'tortas'

        WHEN LOWER(descrip) LIKE '%pizza%' 
          OR LOWER(descrip) LIKE '%pechuguitas%'
          OR LOWER(descrip) LIKE '%frizzio%'  
          OR LOWER(descrip) LIKE '%empanada%' THEN 'comidas'

        WHEN LOWER(descrip) LIKE '%tent%' THEN 'helado_1l'
        
        WHEN LOWER(descrip) LIKE '%1 kg%' 
          OR LOWER(descrip) LIKE '%1/2%'
          OR LOWER(descrip) LIKE '%1/4%'
          OR LOWER(descrip) LIKE '%kilo%'
          OR LOWER(descrip) LIKE '%familiar%' THEN 'granel'

        WHEN LOWER(descrip) LIKE '%pal%' 
          OR LOWER(descrip) LIKE '%cremoso%'
          OR LOWER(descrip) LIKE '%frutilla%'
          OR LOWER(descrip)LIKE '%lim%'
          OR LOWER(descrip) LIKE '%palito%' THEN 'palitos'

        WHEN LOWER(descrip) LIKE '%bombon%' 
          OR LOWER(descrip) LIKE '%alfajor%'
          OR LOWER(descrip) LIKE 'b%' THEN 'bombones'

        WHEN LOWER(descrip) LIKE '%cucur%' 
          OR LOWER(descrip) LIKE '%tacita%' 
          OR LOWER(descrip)LIKE '%bocha%' 
          OR LOWER(descrip)LIKE '%vasito%' 
          OR LOWER(descrip)LIKE '%gigante%'
          OR LOWER(descrip)LIKE '%gridito%'
          OR LOWER(descrip)LIKE '%grido%'
          OR LOWER(descrip)LIKE '%cups%' THEN 'helado_bocha'

        WHEN LOWER(descrip) LIKE '%smoothie%' 
          OR LOWER(descrip) LIKE '%batido%' THEN 'bebidas'

        WHEN LOWER(descrip) LIKE '%yogurt%' THEN 'yogurt_helado'

        WHEN LOWER(descrip) LIKE '%veg%' THEN 'vegano'

        WHEN LOWER(descrip) LIKE '%doble c%' THEN 'bocaditos'

         WHEN LOWER(descrip) LIKE '%hel dul%' 
            OR LOWER(descrip)LIKE '%helado dulzura%' THEN 'helado_sin_azucar'

        WHEN LOWER(descrip) LIKE '%almendrado%'
          OR LOWER(descrip)LIKE '%delicia%'
          OR LOWER(descrip)LIKE '%crocantino%' 
          OR LOWER(descrip) LIKE '%casatta%' THEN 'postres_helados'

        WHEN LOWER(descrip) LIKE '%sundae%'
          OR LOWER(descrip)LIKE '%go%'
          OR LOWER(descrip)LIKE '%shot%'
          OR LOWER(descrip)LIKE '%dulce de leche%'
          OR LOWER(descrip)LIKE '%mermelad%'
          OR LOWER(descrip)LIKE '%shot%'
          OR LOWER(descrip) LIKE '%capelina%' THEN 'especiales'

        ELSE 'otros'
    END AS categoria
FROM catalog_ventas.silver.ventas_clean
)
SELECT 
*
FROM base
WHERE categoria='otros'
GROUP BY articulo, descrip, precio, cliente,categoria

In [0]:

%sql
--delicia y crocantino: postre 
helado dulzura (frutilla, duraznop y chocolate): helado sin azucar
helado por bocha : gigante de 3 b mini gigante 1 b super gridito grido toy 2da edic
Especiales: sundae go shot, capelina nuez, capelina frutilla

In [0]:
%sql
select articulo, descrip from catalog_ventas.silver.ventas_clean
group by articulo, descrip

In [0]:
%sql
with ventas_dias as (
select
  date_format(vtafecha, 'HH:mm') AS vtafecha,
  usulogin,
  sum(total) AS total
from catalog_ventas.silver.ventas_clean
group by vtafecha, usulogin
)
select
vtafecha,
usulogin,
total,
rank() over(partition by vtafecha order by total desc) as ranking
from ventas_dias
order by vtafecha, ranking



In [0]:
%sql
with resumen as (
select
  turno,
  articulo,
  sum(total) as total,
  sum(cant) as unidades
from catalog_ventas.silver.ventas_clean
group by turno, articulo
)

select
turno,
articulo,
total,
unidades,
round(total / sum(total) over(partition by turno)* 100.00,2) as porcentaje
from resumen
order by turno, total desc


In [0]:
%sql
select
turno,
sum(total) as facturacion_total,
round(sum(total) / sum(sum(total)) over()* 100.00,2) as pct_facturacion
from catalog_ventas.silver.ventas_clean
group by turno
order by facturacion_total desc

In [0]:
%sql

CREATE OR REPLACE TEMP VIEW stagin_zona AS
SELECT * FROM VALUES
('capital-federal', 'CABA', 'Buenos-Aires', 'Argentina'),
('palermo', 'CABA', 'Buenos-Aires', 'Argentina'),
('nueva-xona-test', 'GBA', 'Buenos-Aires', 'Argentina')
AS (zona, ciudad, provincia, pais)

In [0]:
%sql
SELECT COUNT(*) AS total_antes FROM stagin_zona